In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize,poly)
from sklearn.model_selection import train_test_split
from functools import partial
from sklearn.model_selection import \
(cross_validate,KFold,ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
import statsmodels.formula.api as smf

In [12]:
np.random.seed(1)
Default=load_data('Default')

In [14]:
Default.default

0       No
1       No
2       No
3       No
4       No
        ..
9995    No
9996    No
9997    No
9998    No
9999    No
Name: default, Length: 10000, dtype: category
Categories (2, object): ['No', 'Yes']

In [20]:
allvars = Default.columns.drop(['default','student'])
design=MS(allvars)
X=design.fit_transform(Default)
y=Default.default=='Yes'
glm = sm.GLM(y,X,family=sm.families.Binomial())
res=glm.fit()
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                default   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -789.48
Date:                Mon, 05 May 2025   Deviance:                       1579.0
Time:                        19:42:48   Pearson chi2:                 6.95e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1256
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept    -11.5405      0.435    -26.544      0.000     -12.393     -10.688
balance        0.0056      0.000     24.835      0.000       0.005       0.006
income      2.081e-05   4.99e-06      4.174      0.000     1.1e-05    3.06e-05
==============================================================================
"""

In [24]:
Default_train, Default_valid = train_test_split(Default, test_size=5000, random_state=0)
Default_valid

,default,student,balance,income
9394,No,Yes,0.000000,13911.441282
898,No,No,396.985985,55454.631040
2398,No,No,1046.416673,47598.307369
5906,No,No,836.343137,34559.158405
2343,No,Yes,534.692907,18729.566240
...,...,...,...,...
3996,No,No,794.176213,43335.985041
5889,No,No,81.531865,40847.811311
4577,No,No,1137.791157,21103.429621
8600,No,No,646.081240,36686.959447


In [28]:
allvars_train = Default_train.columns.drop(['default','student'])
design_train = MS(allvars_train)
X_train = design_train.fit_transform(Default_train)
y_train = Default_train.default == 'Yes'
glm = sm.GLM(y_train, X_train,family=sm.families.Binomial())
res=glm.fit()
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                default   No. Observations:                 5000
Model:                            GLM   Df Residuals:                     4997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -364.78
Date:                Mon, 05 May 2025   Deviance:                       729.56
Time:                        19:50:14   Pearson chi2:                 2.57e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1125
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept    -11.3896      0.635    -17.935      0.000     -12.634     -10.145
balance        0.0056      0.000     16.792      0.000       0.005       0.006
income       1.58e-05   7.34e-06      2.151      0.031     1.4e-06    3.02e-05
==============================================================================
"""

In [34]:
X_valid = design_train.transform(Default_valid)
valid_pred = res.predict(X_valid)
labels = np.array(['No']*5000)
labels[valid_pred>0.5]='Yes'

In [36]:
y_valid = Default_valid['default']
1 - accuracy_score(y_valid, labels)

0.03979999999999995

In [40]:
Default['default_yes'] = (Default['default'] == 'Yes').astype('int')
allvars = Default.columns.drop(['default','student','default_yes'])
design = MS(allvars)
X = design.fit_transform(Default)
y = Default.default == 'Yes'
model =sm.GLM(y,X,family = sm.families.Binomial())
result = model.fit()
summarize(result)

,coef,std err,z,P>|z|
intercept,-11.540500,0.435000,-26.544,0.0
balance,0.005600,0.000000,24.835,0.0
income,0.000021,0.000005,4.174,0.0


In [42]:
result.bse

intercept    0.434772
balance      0.000227
income       0.000005
dtype: float64

In [63]:
def boost_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]

    for _ in range(B):
        idx = rng.choice(D.index, n, replace=True)
        value = func(D, idx)  # ✅ 함수 호출 결과값 받아오기
        first_ += value
        second_ += value**2

    return np.sqrt(second_ / B - (first_ / B)**2)
def boost_fn(model_matrix,response,D,idx):
    D_ = D.loc[idx]
    y = D_[response]
    X = clone(model_matrix).fit_transform(D_)
    model = sm.GLM(y,X,family = sm.families.Binomial())
    result = model.fit()
    coef_income = result.params.iloc[1]
    coef_balance = result.params.iloc[2]
    return result.params

In [65]:
hp_func=partial(boost_fn,MS(['income','balance']),'default_yes')
rng = np.random.default_rng(0)
np.array([hp_func(Default,rng.choice(392,392,replace=True))for _ in range(10)])

array([[-1.37418699e+01,  4.46108654e-05,  6.54456450e-03],
       [-9.73343593e+00,  1.29161125e-05,  4.36172555e-03],
       [-1.09652934e+01,  6.88991070e-06,  5.60127348e-03],
       [-1.29994735e+01,  5.80207964e-05,  5.99463447e-03],
       [-1.76482871e+01,  3.85717484e-05,  8.94706073e-03],
       [-1.37799054e+01,  7.20354708e-05,  5.64428958e-03],
       [-1.00286504e+01,  1.25804073e-05,  5.16837631e-03],
       [-1.36627122e+01,  3.52359713e-05,  6.88246994e-03],
       [-1.30954565e+01,  8.56134990e-06,  7.38734450e-03],
       [-1.41538872e+01,  3.45878793e-05,  7.22152035e-03]])

In [68]:
boost_SE(hp_func,Default,B=1000,seed=10)

intercept    0.425280
income       0.000005
balance      0.000227
dtype: float64